**Auteur(s)** : Cheikhou Akhmed KANE

**Description** : Mise en place `contourZH.shp` et `donneesMNT_ZH.csv`

# Mise en place `contourZH.shp` et `donneesMNT_ZH.csv`

In [1]:
import geopandas as gpd
from pathlib import Path

# --- 1. CHEMINS ---
base_dir = Path.cwd().parent.resolve()

# Fichier en entrée
input_parcellaire_path = base_dir / "data" / "sols" / "shapefiles" / "processed" / "parcellaire_enrichi.shp"

# Fichier en sortie
output_contour_path = base_dir / "includes_sassemeV1" / "modeleHydrographique" / "zonesHydrographiques" / "contourZH.shp"


# --- 2. CHARGEMENT ET VÉRIFICATION DU CRS ---
try:
    gdf_parcellaire = gpd.read_file(input_parcellaire_path)
    print(f"✅ Parcellaire chargé avec {len(gdf_parcellaire)} polygones.")
    
    # Vérification cruciale pour le calcul de surface
    if not gdf_parcellaire.crs.is_projected:
        raise TypeError("Le CRS du shapefile doit être projeté (ex: UTM) pour calculer des surfaces en mètres.")
    print(f"   -> CRS détecté : {gdf_parcellaire.crs.name}. Unités en mètres confirmées.")

except Exception as e:
    print(f"🚨 ERREUR : {e}")


# --- 3. FUSION ET CALCUL DE SURFACE ---
# 3a. Fusionner tous les polygones en une seule géométrie
contour_poly = gdf_parcellaire.unary_union
print("\n-> Toutes les parcelles ont été fusionnées en un seul polygone.")

# 3b. Créer un GeoDataFrame temporaire avec ce polygone unique
gdf_contour = gpd.GeoDataFrame(geometry=[contour_poly], crs=gdf_parcellaire.crs)

# 3c. Calculer les surfaces
surface_m2 = gdf_contour.geometry.area.iloc[0]
surface_ha = surface_m2 / 10000
print(f"-> Surface calculée : {surface_m2:.2f} m² ({surface_ha:.2f} ha)")


# --- 4. CRÉATION DU GEODATAFRAME FINAL ---
# On ajoute les attributs requis
gdf_contour['Code_Zone'] = 'SSM1'
gdf_contour['Surface'] = surface_m2
gdf_contour['Area_ha'] = surface_ha

# On ordonne les colonnes
gdf_contour = gdf_contour[['Code_Zone', 'Surface', 'Area_ha', 'geometry']]
print("✅ GeoDataFrame final créé.")


# --- 5. SAUVEGARDE ---
output_contour_path.parent.mkdir(parents=True, exist_ok=True)
gdf_contour.to_file(output_contour_path, driver='ESRI Shapefile', encoding='utf-8')
print(f"\n✅ Fichier 'contourZH.shp' sauvegardé dans :\n   {output_contour_path}")


# --- 6. VÉRIFICATION ---
print("\nAperçu du fichier final :")
display(gdf_contour)

✅ Parcellaire chargé avec 749 polygones.
   -> CRS détecté : WGS 84 / UTM zone 28N. Unités en mètres confirmées.


C:\Users\Cheikhou\AppData\Local\Temp\ipykernel_26692\914964686.py:30: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  contour_poly = gdf_parcellaire.unary_union



-> Toutes les parcelles ont été fusionnées en un seul polygone.
-> Surface calculée : 2254072.35 m² (225.41 ha)
✅ GeoDataFrame final créé.

✅ Fichier 'contourZH.shp' sauvegardé dans :
   C:\Users\Cheikhou\Desktop\Ferlo_Sine\maelia-data-diohine-v1\includes_sassemeV1\modeleHydrographique\zonesHydrographiques\contourZH.shp

Aperçu du fichier final :


,Code_Zone,Surface,Area_ha,geometry
0,SSM1,2.254072e+06,225.407235,"MULTIPOLYGON (((333627.355 1599982.626, 333629..."


## Copie du Fichier de Données MNT

Pour finaliser la préparation du `modeleHydrographique`, nous copions le fichier `donneesMNT_ZH.csv`.

La cellule de code suivante assure sa copie depuis le dossier de données brutes (`data/`) vers son emplacement final requis par MAELIA.

In [2]:
import shutil

# --- CHEMINS POUR LA COPIE ---
source_mnt_path = base_dir / "data" / "hydro" / "csv" / "raw" / "donneesMNT_ZH.csv"
destination_mnt_path = base_dir / "includes_sassemeV1" / "modeleHydrographique" / "zonesHydrographiques" / "donneesMNT_ZH.csv"

# --- PROCESSUS DE COPIE ---
try:
    # S'assurer que le dossier de destination existe
    destination_mnt_path.parent.mkdir(parents=True, exist_ok=True)
    
    # Copier le fichier
    shutil.copy(source_mnt_path, destination_mnt_path)
    
    print("\n✅ Fichier 'donneesMNT_ZH.csv' copié avec succès.")
    print(f"   Source : {source_mnt_path}")
    print(f"   Destination : {destination_mnt_path}")

except FileNotFoundError:
    print(f"🚨 ERREUR : Le fichier source n'a pas été trouvé. Vérifiez le chemin :")
    print(f"   {source_mnt_path}")
except Exception as e:
    print(f"🚨 Une erreur est survenue : {e}")


✅ Fichier 'donneesMNT_ZH.csv' copié avec succès.
   Source : C:\Users\Cheikhou\Desktop\Ferlo_Sine\maelia-data-diohine-v1\data\hydro\csv\raw\donneesMNT_ZH.csv
   Destination : C:\Users\Cheikhou\Desktop\Ferlo_Sine\maelia-data-diohine-v1\includes_sassemeV1\modeleHydrographique\zonesHydrographiques\donneesMNT_ZH.csv
